In [ ]:

import os
import time
import json
import requests
import pandas as pd
import numpy as np
from datetime import timedelta
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import precision_score, recall_score
from dotenv import load_dotenv
import joblib
import warnings
warnings.filterwarnings('ignore')

In [ ]:
"""
ETH Whale Data Pipeline - Dune + CoinGecko Caching
Handles incremental data fetching, caching, and updates
"""

import os, time, json, requests, warnings
import pandas as pd
import numpy as np
from datetime import timedelta
from dotenv import load_dotenv

warnings.filterwarnings('ignore')
load_dotenv()

DUNE_API_KEY = os.getenv("DUNE_WHALES_API")
COINGECKO_API_KEY = os.getenv("COINGECKO_API_KEY")

os.makedirs("data", exist_ok=True)
os.makedirs("data/price_cache", exist_ok=True)

QUERIES = {
    "whales": ("6395391", "data/dune_whales_cache.json", "data/whale_ml_ready.csv"),
    "market_intent": ("6385600", "data/dune_intent_cache.json", "data/market_intent_ml_ready.csv")
}

# ✅ FIX 1: Date ranges
COINGECKO_START = pd.Timestamp('2017-05-01', tz='UTC')  # Earlier than Dune
DUNE_START = pd.Timestamp('2017-10-16', tz='UTC')  # Earliest clean ETH data

PRICE = [
    'eth_ret_lag1','eth_ret_lag3','eth_ret_lag7',
    'eth_vol7','eth_vol30','eth_rsi',
    'btc_ret_lag1','btc_ret_lag3','btc_ret_lag7',
    'btc_vol7','btc_vol30','btc_rsi',
    'eth_btc_ratio','eth_btc_ratio_ma7','eth_btc_corr_30d'
]

ONCHAIN = [
    'whale_tx_zscore_90d','whale_volume_ratio',
    'whale_volume_ratio_delta_1d','whale_volume_ratio_delta_3d',
    'exchange_flow_share','net_exchange_flow_ratio',
    'whale_exchange_flow_ratio','whale_exchange_asymmetry',
    'tx_per_active_zscore_90d','eth_burned_zscore_90d'
]


# Define separate feature sets for LONG vs SHORT
SHORT_FEATURES = [
    # Distribution signals (funds leaving wallets)
    'exchange_flow_share',
    'net_exchange_flow_ratio',
    'whale_exchange_flow_ratio',
    'whale_exchange_asymmetry',
    
    # Volatility expansion (bear moves are volatile)
    'eth_vol7',
    'eth_vol30',
    
    # Macro pressure
    'btc_ret_lag1',
    'btc_ret_lag3',
    'btc_ret_lag7',
    
    # Correlation breakdown
    'eth_btc_corr_30d',
    'eth_btc_ratio',
    
    # Whale behavior changes
    'whale_volume_ratio_delta_3d',
    'whale_tx_zscore_90d'
]

LONG_FEATURES = [
    # Use full feature set for LONG (accumulation is complex)
    'btc_rsi', 'eth_vol7', 'whale_volume_ratio', 'eth_rsi',
    'btc_ret_lag1', 'eth_burned_zscore_90d', 'eth_btc_corr_30d',
    'eth_ret_lag1', 'btc_ret_lag7', 'btc_vol30',
    'whale_volume_ratio_delta_1d', 'whale_volume_ratio_delta_3d',
    'exchange_flow_share', 'net_exchange_flow_ratio',
    'whale_exchange_flow_ratio', 'tx_per_active_zscore_90d'
]

def fetch_dune(qid, cache):
    """Incrementally fetch Dune data, only updating missing dates"""
    headers = {"x-dune-api-key": DUNE_API_KEY}
    today = pd.Timestamp.now(tz='UTC').normalize()
    yesterday = today - timedelta(1)
    
    if os.path.exists(cache):
        with open(cache) as f:
            c = json.load(f)
        df_cached = pd.DataFrame(c["data"])
        df_cached["block_date"] = pd.to_datetime(df_cached["block_date"], utc=True)
        last_date = pd.to_datetime(c["last_block_date"], utc=True)
        
        if last_date >= yesterday:
            print(f"✅ {os.path.basename(cache)} current ({last_date.date()})")
            return df_cached
    else:
        df_cached = pd.DataFrame()
    
    resp = requests.post(f"https://api.dune.com/api/v1/query/{qid}/execute", 
                        headers=headers, timeout=30).json()
    eid = resp["execution_id"]
    
    for _ in range(60):
        status = requests.get(f"https://api.dune.com/api/v1/execution/{eid}/status", 
                            headers=headers).json()["state"]
        if status == "QUERY_STATE_COMPLETED": 
            break
        time.sleep(10)
    
    result = requests.get(f"https://api.dune.com/api/v1/execution/{eid}/results", 
                         headers=headers).json()["result"]["rows"]
    df_new = pd.DataFrame(result)
    df_new["block_date"] = pd.to_datetime(df_new["block_date"], utc=True)
    
    df = pd.concat([df_cached, df_new[df_new["block_date"] < today]]) \
        .drop_duplicates("block_date").sort_values("block_date").reset_index(drop=True)
    
    with open(cache, "w") as f:
        json.dump({
            "last_block_date": df["block_date"].max().strftime("%Y-%m-%d"), 
            "data": json.loads(df.to_json(orient="records"))
        }, f)
    
    return df

def to_utc(ts):
    """Ensure timestamp is UTC"""
    ts = pd.Timestamp(ts)
    return ts.tz_localize("UTC") if ts.tzinfo is None else ts.tz_convert("UTC")

def fetch_cg_chunked(cg_id, start, end, key, days=30):
    """Fetch CoinGecko prices in chunks to avoid rate limits"""
    url = "https://pro-api.coingecko.com/api/v3"
    headers = {"x-cg-pro-api-key": key}
    start_dt, end_dt = to_utc(start), to_utc(end) + pd.Timedelta(days=1)
    all_prices, curr = [], start_dt
    
    while curr < end_dt:
        next_dt = min(curr + pd.Timedelta(days=days), end_dt)
        params = {"vs_currency": "usd", "from": int(curr.timestamp()), 
                 "to": int(next_dt.timestamp())}
        r = requests.get(f"{url}/coins/{cg_id}/market_chart/range", 
                        params=params, headers=headers, timeout=30)
        all_prices.extend(r.json().get("prices", []))
        time.sleep(0.3)
        curr = next_dt
    
    df = pd.DataFrame(all_prices, columns=["timestamp", "price"])
    df["date"] = pd.to_datetime(df["timestamp"], unit="ms", utc=True).dt.floor("D")
    return df.groupby("date")["price"].mean().reset_index()

def get_price(sym, cg_id, start, end, key):
    """Incrementally fetch price data, only updating missing dates"""
    cache = f"data/price_cache/{sym}.csv"
    today_utc = pd.Timestamp.utcnow().floor("D")
    yesterday = today_utc - pd.Timedelta(days=1)
    start, end = to_utc(start), min(to_utc(end), yesterday)
    
    if os.path.exists(cache):
        df = pd.read_csv(cache, parse_dates=["date"])
        df["date"] = df["date"].apply(to_utc)
        if df["date"].max() >= end:
            print(f"✅ {sym.upper()} cache current")
            return df
        
        new = fetch_cg_chunked(cg_id, df["date"].max() + pd.Timedelta(days=1), end, key)
        df = pd.concat([df, new.rename(columns={"price": f"{sym}_price"})]) \
            .drop_duplicates("date").reset_index(drop=True)
    else:
        df = fetch_cg_chunked(cg_id, start, end, key) \
            .rename(columns={"price": f"{sym}_price"})
    
    df.to_csv(cache, index=False)
    return df

def load_all_data():
    """Load all data sources with incremental updates"""
    datasets = {}
    for name, (qid, cache, output) in QUERIES.items():
        datasets[name] = fetch_dune(qid, cache)
        datasets[name].to_csv(output, index=False)
        time.sleep(0.5)
    
    max_date = max(datasets["whales"]["block_date"].max(), 
                   datasets["market_intent"]["block_date"].max())
    df_btc = get_price("btc", "bitcoin", COINGECKO_START, max_date, COINGECKO_API_KEY)
    df_eth = get_price("eth", "ethereum", COINGECKO_START, max_date, COINGECKO_API_KEY)
    
    return datasets["whales"], datasets["market_intent"], df_btc, df_eth

In [ ]:
# ========== FEATURE ENGINEERING ==========
def rolling_zscore(s, w=90):
    return ((s - s.rolling(w).mean()) / s.rolling(w).std()).shift(1)

def add_price_features(df, price_col, prefix):
    df[f'{prefix}_log_return'] = np.log(df[price_col] / df[price_col].shift(1))
    
    for lag in [1, 3, 7]:
        df[f'{prefix}_ret_lag{lag}'] = df[f'{prefix}_log_return'].shift(lag)
    
    df[f'{prefix}_vol7'] = df[f'{prefix}_log_return'].rolling(7).std().shift(1)
    df[f'{prefix}_vol30'] = df[f'{prefix}_log_return'].rolling(30).std().shift(1)
    
    ret = df[f'{prefix}_log_return']
    gains = ret.where(ret > 0, 0).rolling(14).mean()
    losses = -ret.where(ret < 0, 0).rolling(14).mean()
    df[f'{prefix}_rsi'] = (100 - (100 / (1 + gains / (losses + 1e-10)))).shift(1)
    
    return df

def engineer_features(df_whales, df_market_intent, df_btc, df_eth):
    df_prices = pd.merge(df_btc, df_eth, on='date', how='outer').sort_values('date')
    df = pd.merge(df_whales, df_prices, left_on='block_date', right_on='date', how='left').drop(columns=['date'])
    df = pd.merge(df, df_market_intent, on='block_date', how='left', suffixes=('', '_intent'))
    df = df.sort_values('block_date').reset_index(drop=True)
    
    df = add_price_features(df, 'eth_price', 'eth')
    df = add_price_features(df, 'btc_price', 'btc')
    
    df['eth_btc_ratio'] = df['eth_price'] / df['btc_price']
    df['eth_btc_ratio_ma7'] = df['eth_btc_ratio'].rolling(7).mean().shift(1)
    df['eth_btc_corr_30d'] = df['eth_log_return'].shift(1).rolling(30).corr(df['btc_log_return'].shift(1)).shift(1)
    
    # On-chain z-scores
    for raw, zscore in [('whale_tx_count', 'whale_tx_zscore_90d'), ('tx_per_active', 'tx_per_active_zscore_90d'), ('eth_burned', 'eth_burned_zscore_90d')]:
        if raw in df.columns:
            df[zscore] = rolling_zscore(df[raw], 90)
    
    # ✅ FIX C: Exchange volume z-score
    if 'exchange_volume' in df.columns:
        df['exchange_volume_zscore'] = rolling_zscore(df['exchange_volume'], 90)
    
    # Burn/issuance ratio
    if 'eth_burned' in df.columns and 'total_gas_fees' in df.columns:
        df['burn_issuance_ratio'] = (df['eth_burned'] / (df['total_gas_fees'] + 1e-10)).shift(1)
    
    # Whale deltas
    if 'whale_volume_ratio' in df.columns:
        df['whale_volume_ratio_delta_1d'] = df['whale_volume_ratio'].diff(1).shift(1)
        df['whale_volume_ratio_delta_3d'] = df['whale_volume_ratio'].diff(3).shift(1)
    
    df = df.drop(columns=['eth_log_return', 'btc_log_return'], errors='ignore')
    df.to_csv('data/features_engineered.csv', index=False)
    print(f"✅ Features (ALL LAGGED): {len(df.columns)} cols")
    return df

# ========== TARGETS (✅ FIX A: TWO-TIER SHORT) ==========
def create_targets_two_tier(df, k=1.5):
    """✅ FIX A: Two-tier SHORT labels (crash + breakdown)"""
    df = df.sort_values('block_date').reset_index(drop=True)
    df['eth_log_return'] = np.log(df['eth_price'] / df['eth_price'].shift(1))
    df['rolling_vol_30'] = df['eth_log_return'].rolling(30, min_periods=10).std()
    
    df['return_t2'] = df['eth_log_return'].rolling(2).sum().shift(-2)
    df['threshold_t2'] = df['rolling_vol_30'].rolling(30, min_periods=10).median() * k
    
    # ✅ FIX A: Tier 1 - Crash (hard down)
    hard_down = (
        (df['return_t2'] < -df['threshold_t2']) &
        (df['eth_vol7'] > df['eth_vol30']).fillna(False)
    )
    
    # ✅ FIX A: Tier 2 - Breakdown (pre-crash)
    exchange_flow_median = df['exchange_flow_share'].rolling(90, min_periods=30).median()
    
    soft_down = (
        (df['eth_ret_lag1'].fillna(0) < 0) &
        (df['btc_ret_lag1'].fillna(0) < 0) &
        (df['whale_volume_ratio_delta_3d'].fillna(0) > 0) &
        (df['exchange_flow_share'] > exchange_flow_median).fillna(False)
    )
    
    # Create targets
    df['target_t2'] = 0
    df.loc[df['return_t2'] > df['threshold_t2'], 'target_t2'] = 1  # UP
    df.loc[hard_down | soft_down, 'target_t2'] = -1  # DOWN (both tiers)
    
    df['y_long_t2'] = (df['target_t2'] == 1).astype(int)
    df['y_short_t2'] = (df['target_t2'] == -1).astype(int)
    
    df = df.drop(columns=['eth_log_return'], errors='ignore')
    
    print(f"\n✅ Two-Tier SHORT Targets (k={k}):")
    for state, label in [(-1, 'DOWN'), (0, 'FLAT'), (1, 'UP')]:
        count = (df['target_t2'] == state).sum()
        print(f"  {label:5s}: {count:4d} ({count/len(df)*100:5.1f}%)")
    
    hard_count = hard_down.sum()
    soft_count = soft_down.sum()
    total_down = (df['target_t2'] == -1).sum()
    print(f"\n  Tier 1 (crash):     {hard_count:4d}")
    print(f"  Tier 2 (breakdown): {soft_count:4d}")
    print(f"  Total DOWN:         {total_down:4d}")
    
    return df

In [ ]:
# ========== REGIMES (✅ FIX E: R5 DISTRIBUTION) ==========
def define_regimes_extended(df):
    """✅ FIX E: Add R5 distribution regime"""
    if 'btc_ret_lag1' not in df.columns or 'eth_vol7' not in df.columns:
        df['regime_code'] = 'R0'
        return df
    
    # Standard regimes
    btc_trend_7d = df['btc_ret_lag1'].rolling(7).mean()
    df['btc_regime'] = pd.cut(btc_trend_7d, bins=[-np.inf, -0.005, 0.005, np.inf], labels=['DOWN', 'FLAT', 'UP'])
    
    vol_median = df['eth_vol7'].rolling(180, min_periods=60).median()
    df['vol_regime'] = (df['eth_vol7'] > vol_median).map({True: 'HIGH', False: 'LOW'})
    
    df['regime'] = df['btc_regime'].astype(str) + '_' + df['vol_regime'].astype(str)
    regime_map = {'UP_HIGH': 'R1', 'UP_LOW': 'R2', 'DOWN_HIGH': 'R3', 'DOWN_LOW': 'R4'}
    df['regime_code'] = df['regime'].map(regime_map).fillna('R0')
    
    # ✅ FIX E: Whale distribution regime (R5)
    exchange_flow_median = df['exchange_flow_share'].rolling(60, min_periods=20).median()
    
    df['dist_regime'] = (
        (df['whale_volume_ratio_delta_3d'].fillna(0) > 0) &
        (df['exchange_flow_share'] > exchange_flow_median).fillna(False)
    )
    
    df.loc[df['dist_regime'], 'regime_code'] = 'R5'
    
    print(f"\n✅ Extended Regime Distribution:")
    for code in ['R1', 'R2', 'R3', 'R4', 'R5', 'R0']:
        count = (df['regime_code'] == code).sum()
        pct = (count / len(df)) * 100 if len(df) > 0 else 0
        icon = '🟢' if code == 'R1' else ('🔴' if code in ['R3', 'R5'] else '⚪')
        print(f"  {icon} {code}: {count:4d} ({pct:5.1f}%)")
    
    return df

# ========== MODEL TRAINING (✅ FIX B, D) ==========
def compress_features(X, y, features, top_n=10):
    model = GradientBoostingClassifier(n_estimators=50, max_depth=3, random_state=42)
    X_sub = X[features].fillna(method='ffill').fillna(0)
    model.fit(X_sub, y)
    
    importances = pd.DataFrame({'feature': features, 'importance': model.feature_importances_}).sort_values('importance', ascending=False)
    return importances.head(top_n)['feature'].tolist()

def prepare_regime_data(df, regime, direction, features):
    target_col = f'y_{direction.lower()}_t2'
    regime_data = df[df['regime_code'] == regime].copy()
    
    # ✅ FIX B: Remove over-restrictive vol filter
    if direction == 'SHORT':
        if 'btc_regime' in regime_data.columns and 'vol_regime' in regime_data.columns:
            regime_data = regime_data[
                (regime_data['btc_regime'].isin(['DOWN', 'FLAT'])) &
                (regime_data['vol_regime'].isin(['HIGH', 'LOW']))  # ✅ BOTH vol regimes
            ]
    
    regime_data = regime_data[regime_data['target_t2'] != 0]
    features = [f for f in features if f in regime_data.columns]
    
    X = regime_data[features].fillna(method='ffill').fillna(0)
    y = regime_data[target_col]
    
    return X, y

def train_regime_model(X, y, name, is_short=False):
    if len(X) < 30: return None, None
    
    split = int(len(X) * 0.8)
    X_train, X_val = X.iloc[:split], X.iloc[split:]
    y_train, y_val = y.iloc[:split], y.iloc[split:]
    
    model = GradientBoostingClassifier(n_estimators=150, max_depth=4, learning_rate=0.05, random_state=42)
    model.fit(X_train, y_train)
    
    y_prob = model.predict_proba(X_val)[:, 1]
    
    # ✅ FIX D: Lower precision gate for SHORT
    min_prec = 0.55 if is_short else 0.65
    best_thresh, best_prec = 0.65, 0
    
    for thresh in np.arange(0.55, 0.80, 0.05):
        y_pred = (y_prob >= thresh).astype(int)
        if y_pred.sum() > 0:
            prec = precision_score(y_val, y_pred, zero_division=0)
            if prec >= min_prec and prec > best_prec:
                best_prec, best_thresh = prec, thresh
    
    y_pred = (y_prob >= best_thresh).astype(int)
    prec = precision_score(y_val, y_pred, zero_division=0)
    rec = recall_score(y_val, y_pred, zero_division=0)
    
    print(f"   Prec={prec:.3f}, Rec={rec:.3f}, Thresh={best_thresh:.2f}")
    return model, best_thresh

def train_all_models(df):
    models, thresholds = {}, {}
    
    # R1 LONG
    print(f"\n🟢 R1 LONG")
    long_feats = [f for f in LONG_FEATURES if f in df.columns]
    X, y = prepare_regime_data(df, 'R1', 'LONG', long_feats)
    
    if len(X) >= 50:
        top = compress_features(X, y, long_feats, 10)
        model, thresh = train_regime_model(X[top], y, 'R1_LONG', is_short=False)
        if model:
            models['R1_LONG'] = {'model': model, 'features': top}
            thresholds['R1_LONG'] = thresh
    
    # R3 SHORT (existing bear regime)
    print(f"\n🔴 R3 SHORT (TWO-TIER)")
    short_feats = [f for f in SHORT_FEATURES if f in df.columns]
    X, y = prepare_regime_data(df, 'R3', 'SHORT', short_feats)
    
    if len(X) >= 50:
        top = compress_features(X, y, short_feats, 8)
        model, thresh = train_regime_model(X[top], y, 'R3_SHORT', is_short=True)
        if model:
            models['R3_SHORT'] = {'model': model, 'features': top}
            thresholds['R3_SHORT'] = thresh
    
    # ✅ FIX E: R5 SHORT (distribution regime)
    print(f"\n🔴 R5 SHORT (DISTRIBUTION)")
    X, y = prepare_regime_data(df, 'R5', 'SHORT', short_feats)
    
    if len(X) >= 50:
        top = compress_features(X, y, short_feats, 8)
        model, thresh = train_regime_model(X[top], y, 'R5_SHORT', is_short=True)
        if model:
            models['R5_SHORT'] = {'model': model, 'features': top}
            thresholds['R5_SHORT'] = thresh
    
    return models, thresholds


In [ ]:
# ========== VETO SYSTEM ==========
class ProductionEngine:
    def __init__(self, models, thresholds):
        self.models = models
        self.thresholds = thresholds
    
    def predict(self, row, regime):
        tradeable = {'R1': 'LONG', 'R3': 'SHORT', 'R5': 'SHORT'}
        if regime not in tradeable:
            return {'action': 'NO_TRADE', 'confidence': 0.0, 'veto': ['invalid_regime']}
        
        direction = tradeable[regime]
        key = f'{regime}_{direction}'
        
        if key not in self.models:
            return {'action': 'NO_TRADE', 'confidence': 0.0, 'veto': ['no_model']}
        
        model_info = self.models[key]
        X_sub = row[model_info['features']].fillna(method='ffill').fillna(0)
        prob = model_info['model'].predict_proba(X_sub.values.reshape(1, -1))[0, 1]
        
        veto_reasons = []
        
        # ✅ FIX C: Liquidity-aware veto (now has exchange_volume_zscore)
        whale_ratio = row.get('whale_volume_ratio', 0)
        exchange_vol = row.get('exchange_volume_zscore', 0)
        
        if whale_ratio > 0.30 and exchange_vol < -0.5:
            veto_reasons.append('liquidity_absorbed')
        
        # Standard vetos
        if direction == 'LONG' and row.get('btc_ret_lag1', 0) < -0.02:
            veto_reasons.append('btc_conflict')
        if direction == 'SHORT' and row.get('btc_ret_lag1', 0) > 0.02:
            veto_reasons.append('btc_conflict')
        
        if prob < self.thresholds[key] or veto_reasons:
            return {'action': 'NO_TRADE', 'confidence': prob, 'veto': veto_reasons}
        
        return {'action': direction, 'confidence': prob, 'veto': []}

# ========== DAILY SIGNAL (✅ FIX F: AGGRESSIVE SIZING) ==========
VETO_WEIGHTS = {"btc_conflict": 0.40, "liquidity_absorbed": 0.30, "invalid_regime": 0.50}

def compute_veto_risk(veto_reasons):
    return min(0.9, sum(VETO_WEIGHTS.get(v, 0.0) for v in veto_reasons))

def map_confidence_to_size(conf):
    """✅ FIX F: Aggressive position sizing"""
    if conf < 0.55: return 0.0
    elif conf < 0.60: return 0.25
    elif conf < 0.65: return 0.50
    elif conf < 0.70: return 1.00   # ✅ 0.75 → 1.00
    elif conf < 0.75: return 1.25   # ✅ NEW tier
    else: return 1.50               # ✅ 1.00 → 1.50

def build_daily_signal(row, engine):
    regime = row.get('regime_code', 'R0')
    base = {
        "date": str(row['block_date'].date() if hasattr(row['block_date'], 'date') else row['block_date']),
        "regime": regime,
        "action": "NO_TRADE",
        "position_size": 0.0,
        "veto_reasons": []
    }
    
    if regime not in ['R1', 'R3', 'R5']:
        base["veto_reasons"] = ["invalid_regime"]
        return base
    
    raw = engine.predict(row, regime)
    
    if raw["action"] == "NO_TRADE":
        base["veto_reasons"] = raw.get("veto", [])
        return base
    
    veto_risk = compute_veto_risk(raw.get("veto", []))
    adj_conf = raw["confidence"] * (1 - veto_risk)
    size = map_confidence_to_size(adj_conf)
    
    return {
        "date": base["date"],
        "regime": regime,
        "direction": raw["action"],
        "model_probability": round(raw["confidence"], 3),
        "veto_risk": round(veto_risk, 2),
        "adjusted_confidence": round(adj_conf, 3),
        "position_size": size,
        "action": "ENTER" if size > 0 else "NO_TRADE",
        "veto_reasons": raw.get("veto", [])
    }
# ========== MAIN ==========
def run_complete_pipeline():
    print("="*70)
    print("ETH WHALE PIPELINE - ALL FIXES APPLIED")
    print("="*70)
    
    df_whales, df_intent, df_btc, df_eth = load_all_data()
    df = engineer_features(df_whales, df_intent, df_btc, df_eth)
    df = create_targets_two_tier(df, k=1.5)
    df = define_regimes_extended(df)
    
    models, thresholds = train_all_models(df)
    
    for name, info in models.items():
        joblib.dump(info, f'models/{name}_prod.pkl')
    with open('models/thresholds_prod.json', 'w') as f:
        json.dump(thresholds, f)
    
    engine = ProductionEngine(models, thresholds)
    signal = build_daily_signal(df.iloc[-1], engine)
    
    print(f"\n{'='*70}")
    print("🎯 DAILY SIGNAL")
    print(json.dumps(signal, indent=2))
    print(f"{'='*70}")
    
    with open(f"data/daily_signal_{signal['date']}.json", 'w') as f:
        json.dump(signal, f, indent=2)
    
    df.to_csv('data/pipeline_complete.csv', index=False)
    print("\n✅ COMPLETE - All fixes applied")
    
    return engine, df, signal

if __name__ == "__main__":
    engine, df, signal = run_complete_pipeline()

In [1]:
"""
ETH Whale Production Pipeline - ALL PHASES IMPLEMENTED
"""

import os, json, warnings
import pandas as pd
import numpy as np
from datetime import timedelta
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import precision_score, recall_score, roc_auc_score
import joblib

warnings.filterwarnings('ignore')

SHORT_FEATURES = ['exchange_flow_share','net_exchange_flow_ratio','whale_exchange_flow_ratio',
                  'whale_exchange_asymmetry','eth_vol7','eth_vol30','btc_ret_lag1','btc_ret_lag3',
                  'eth_btc_corr_30d','whale_volume_ratio_delta_3d','exchange_volume_zscore']

SLIPPAGE, FEES = 0.0008, 0.0004

# ========== PHASE 1: WALK-FORWARD VALIDATION ==========
def walk_forward_r5_short(df):
    print("\n" + "="*70)
    print("PHASE 1: WALK-FORWARD VALIDATION (R5 SHORT ISOLATION)")
    print("="*70)
    
    splits = [
        {'train_start':'2017-01-01','train_end':'2020-12-31','test_start':'2021-01-01','test_end':'2021-12-31','name':'2021'},
        {'train_start':'2017-01-01','train_end':'2021-12-31','test_start':'2022-01-01','test_end':'2022-12-31','name':'2022'},
        {'train_start':'2017-01-01','train_end':'2022-12-31','test_start':'2023-01-01','test_end':'2023-12-31','name':'2023'},
        {'train_start':'2017-01-01','train_end':'2023-12-31','test_start':'2024-01-01','test_end':'2024-12-31','name':'2024'},
    ]
    
    results = []
    for split in splits:
        train = df[(df['block_date']>=split['train_start'])&(df['block_date']<=split['train_end'])]
        test = df[(df['block_date']>=split['test_start'])&(df['block_date']<=split['test_end'])]
        
        train_r5, test_r5 = train[train['regime_code']=='R5'], test[test['regime_code']=='R5']
        if len(train_r5)<30 or len(test_r5)<10:
            print(f"⚠️  {split['name']}: Insufficient R5 data")
            continue
        
        features = [f for f in SHORT_FEATURES if f in train_r5.columns]
        X_train = train_r5[features].fillna(method='ffill').fillna(0)
        y_train = train_r5['y_short_t2']
        X_test = test_r5[features].fillna(method='ffill').fillna(0)
        y_test = test_r5['y_short_t2']
        
        model = GradientBoostingClassifier(n_estimators=150,max_depth=4,learning_rate=0.05,random_state=42)
        model.fit(X_train, y_train)
        
        probs = model.predict_proba(X_test)[:,1]
        preds = (probs>0.55).astype(int)
        
        prec = precision_score(y_test,preds,zero_division=0)
        rec = recall_score(y_test,preds,zero_division=0)
        auc = roc_auc_score(y_test,probs) if len(np.unique(y_test))>1 else 0
        
        results.append({'period':split['name'],'precision':prec,'recall':rec,'auc':auc,
                       'train_size':len(train_r5),'test_size':len(test_r5),'signals':preds.sum()})
        
        status = "✅ PASS" if prec>=0.65 and rec>=0.50 else "❌ FAIL"
        print(f"\n{split['name']} {status}")
        print(f"  Precision: {prec:.3f} (req: ≥0.65)")
        print(f"  Recall:    {rec:.3f} (req: ≥0.50)")
        print(f"  AUC:       {auc:.3f}")
        print(f"  Signals:   {preds.sum()}/{len(test_r5)}")
    
    df_results = pd.DataFrame(results)
    df_results.to_csv('validation/walk_forward_r5.csv',index=False)
    
    passed = (df_results['precision']>=0.65).sum()
    print(f"\n{'='*70}")
    print(f"VERDICT: {passed}/{len(df_results)} periods passed")
    print(f"{'='*70}")
    return df_results

# ========== PHASE 2: PNL SIMULATION ==========
def simulate_pnl(df):
    print("\n" + "="*70)
    print("PHASE 2: REGIME-AWARE PNL SIMULATION")
    print("="*70)
    
    df = df.sort_values('block_date').reset_index(drop=True)
    df['price_t2'] = df['eth_price'].shift(-2)
    
    df_r5 = df[df['regime_code']=='R5'].copy()
    features = [f for f in SHORT_FEATURES if f in df_r5.columns]
    
    split = int(len(df_r5)*0.8)
    X_train = df_r5[features].iloc[:split].fillna(method='ffill').fillna(0)
    y_train = df_r5['y_short_t2'].iloc[:split]
    
    model = GradientBoostingClassifier(n_estimators=150,max_depth=4,learning_rate=0.05,random_state=42)
    model.fit(X_train, y_train)
    
    X_all = df[features].fillna(method='ffill').fillna(0)
    df['signal_prob'] = model.predict_proba(X_all)[:,1]
    df['signal'] = ((df['signal_prob']>0.55)&(df['regime_code']=='R5')).astype(int)
    
    trades, cooldown_until = [], None
    for i,row in df.iterrows():
        if pd.isna(row['price_t2']) or pd.isna(row['eth_price']): continue
        if cooldown_until and row['block_date']<=cooldown_until: continue
        
        if row['signal']==1:
            gross_ret = -(row['price_t2']-row['eth_price'])/row['eth_price']
            net_ret = gross_ret-(2*SLIPPAGE)-(2*FEES)
            
            trades.append({'entry_date':row['block_date'],'exit_date':row['block_date']+timedelta(days=2),
                          'regime':row['regime_code'],'entry_price':row['eth_price'],'exit_price':row['price_t2'],
                          'gross_ret':gross_ret,'net_ret':net_ret,'confidence':row['signal_prob']})
            cooldown_until = row['block_date']+timedelta(days=1)
    
    df_trades = pd.DataFrame(trades)
    if len(df_trades)==0:
        print("⚠️  No trades generated")
        return df_trades
    
    print("\n📊 PnL Attribution by Regime:")
    print(df_trades.groupby('regime')['net_ret'].agg(['count','mean','sum','max','min']))
    
    total_ret = df_trades['net_ret'].sum()
    win_rate = (df_trades['net_ret']>0).mean()
    sharpe = df_trades['net_ret'].mean()/(df_trades['net_ret'].std()+1e-10)*np.sqrt(252)
    
    print(f"\n📈 Overall Performance:")
    print(f"  Total trades:  {len(df_trades)}")
    print(f"  Total return:  {total_ret*100:.2f}%")
    print(f"  Win rate:      {win_rate*100:.1f}%")
    print(f"  Sharpe:        {sharpe:.2f}")
    print(f"  Max win:       {df_trades['net_ret'].max()*100:.2f}%")
    print(f"  Max loss:      {df_trades['net_ret'].min()*100:.2f}%")
    
    df_trades.to_csv('backtest/pnl_simulation.csv',index=False)
    return df_trades

# ========== PHASE 3: VETO SCORING ==========
def veto_to_score(row):
    score, reasons = 0, []
    
    if row.get('whale_exchange_flow_ratio',0)>0.6:
        score+=2; reasons.append('whale_exchange_inflow')
    if row.get('net_exchange_flow_ratio',0)<0:
        score+=1; reasons.append('net_flow_negative')
    if row.get('btc_ret_lag1',0)<0:
        score+=1; reasons.append('btc_down')
    if row.get('exchange_volume_zscore',0)>0:
        score+=1; reasons.append('liquidity_high')
    if row.get('eth_vol7',0)>row.get('eth_vol30',0):
        score+=1; reasons.append('vol_expanding')
    
    if row.get('whale_volume_ratio',0)>0.30 and row.get('exchange_volume_zscore',0)<-0.5:
        score-=2; reasons.append('otc_like_flow')
    if row.get('btc_ret_lag1',0)>0.02:
        score-=1; reasons.append('btc_strong_up')
    
    return score, reasons

def build_signal_object(row, model, features, threshold):
    X = row[features].fillna(method='ffill').fillna(0).values.reshape(1,-1)
    prob = model.predict_proba(X)[0,1]
    score, reasons = veto_to_score(row)
    
    if score>=4: action = 'SHORT'
    elif score>=2: action = 'WATCH'
    else: action = 'NO_TRADE'
    
    if prob<threshold: action = 'NO_TRADE'
    
    return {'date':str(row['block_date'].date()),'regime':row['regime_code'],
            'direction':'SHORT' if action=='SHORT' else None,'score':score,
            'confidence':round(prob,3),'expected_horizon':'T+2','action':action,'reasons':reasons}

# ========== PHASE 4: POSITION SIZING ==========
def regime_aware_sizing(confidence, regime):
    if confidence<0.55: base=0.0
    elif confidence<0.60: base=0.4
    elif confidence<0.70: base=0.7
    else: base=1.0
    
    regime_mult = {'R5':1.5,'R3':1.0,'R1':0.75,'R0':0.0}
    return min(base*regime_mult.get(regime,0.0), 1.5)

# ========== PHASE 5: STRESS TESTS ==========
def stress_tests(df):
    print("\n" + "="*70)
    print("PHASE 5: STRESS TESTS")
    print("="*70)
    
    bull = df[(df['block_date']>='2020-01-01')&(df['block_date']<='2021-12-31')]
    bull_signals = bull[bull['regime_code']=='R5']['signal'].sum() if 'signal' in bull.columns else 0
    print(f"\n1️⃣ Bull Market (2020-2021): R5 signals={bull_signals} (expect: low)")
    
    crash1 = df[(df['block_date']>='2020-03-01')&(df['block_date']<='2020-03-31')]
    crash2 = df[(df['block_date']>='2022-11-01')&(df['block_date']<='2022-11-30')]
    print(f"\n2️⃣ Crash Clusters:")
    print(f"  March 2020 R5 days: {(crash1['regime_code']=='R5').sum()}")
    print(f"  Nov 2022 R5 days: {(crash2['regime_code']=='R5').sum()}")
    
    years = df.groupby(df['block_date'].dt.year).apply(lambda x: (x['regime_code']=='R5').sum() if 'signal' not in x.columns else x['signal'].sum())
    print(f"\n3️⃣ Signal Density (yearly):")
    print(years)
    print("✅ Acceptable" if years.max()<=40 else "⚠️  WARNING: Overtrading")

# ========== PHASE 6: DEPLOYMENT CHECKLIST ==========
def deployment_checklist():
    print("\n" + "="*70)
    print("PHASE 6: LIVE DEPLOYMENT CHECKLIST")
    print("="*70)
    
    checks = {'Feature lag integrity':os.path.exists('data/features_engineered.csv'),
              'Data freshness':os.path.exists('data/dune_whales_cache.json'),
              'Model versioning':os.path.exists('models'),'Daily log':os.path.exists('data')}
    
    for check,passed in checks.items():
        print(f"  {'✅' if passed else '❌'} {check}")
    
    print("\n📋 Manual checks:")
    print("  ⚠️  API circuit breaker")
    print("  ⚠️  Kill-switch (3 losses)")

# ========== INTEGRATION ==========
def run_all_phases(df):
    """Execute all 6 phases"""
    for d in ['validation','backtest']:
        os.makedirs(d,exist_ok=True)
    
    # Phase 1
    wf_results = walk_forward_r5_short(df)
    
    # Phase 2
    trades = simulate_pnl(df)
    
    # Phase 3-4: Build final signal
    df_r5 = df[df['regime_code']=='R5'].copy()
    features = [f for f in SHORT_FEATURES if f in df_r5.columns]
    split = int(len(df_r5)*0.8)
    X_train = df_r5[features].iloc[:split].fillna(method='ffill').fillna(0)
    y_train = df_r5['y_short_t2'].iloc[:split]
    
    model = GradientBoostingClassifier(n_estimators=150,max_depth=4,learning_rate=0.05,random_state=42)
    model.fit(X_train, y_train)
    
    latest = df.iloc[-1]
    signal = build_signal_object(latest, model, features, 0.55)
    signal['position_size'] = regime_aware_sizing(signal['confidence'], signal['regime'])
    
    print("\n" + "="*70)
    print("PHASE 3-4: FINAL SIGNAL")
    print(json.dumps(signal,indent=2))
    print("="*70)
    
    # Phase 5
    stress_tests(df)
    
    # Phase 6
    deployment_checklist()
    
    return wf_results, trades, signal

# ========== EXAMPLE USAGE ==========
if __name__ == "__main__":
    # Load your existing pipeline data
    df = pd.read_csv('data/pipeline_complete.csv', parse_dates=['block_date'])
    
    wf_results, trades, signal = run_all_phases(df)
    
    print("\n✅ ALL PHASES COMPLETE")


PHASE 1: WALK-FORWARD VALIDATION (R5 SHORT ISOLATION)

2021 ✅ PASS
  Precision: 0.811 (req: ≥0.65)
  Recall:    0.878 (req: ≥0.50)
  AUC:       0.904
  Signals:   53/112

2022 ✅ PASS
  Precision: 0.898 (req: ≥0.65)
  Recall:    0.815 (req: ≥0.50)
  AUC:       0.960
  Signals:   49/102

2023 ✅ PASS
  Precision: 0.913 (req: ≥0.65)
  Recall:    0.750 (req: ≥0.50)
  AUC:       0.902
  Signals:   23/77

2024 ✅ PASS
  Precision: 0.744 (req: ≥0.65)
  Recall:    0.914 (req: ≥0.50)
  AUC:       0.912
  Signals:   43/92

VERDICT: 4/4 periods passed

PHASE 2: REGIME-AWARE PNL SIMULATION

📊 PnL Attribution by Regime:
        count      mean       sum       max       min
regime                                               
R5        263  0.002141  0.563083  0.208369 -0.174882

📈 Overall Performance:
  Total trades:  263
  Total return:  56.31%
  Win rate:      50.2%
  Sharpe:        0.55
  Max win:       20.84%
  Max loss:      -17.49%

PHASE 3-4: FINAL SIGNAL
{
  "date": "2025-12-30",
  "regime"

In [2]:
"""
ETH WHALE ALPHA PIPELINE - COMPLETE INTEGRATED SOLUTION
All 6 phases implemented with proper data flow
"""

import os
import time
import json
import warnings
import requests
import pandas as pd
import numpy as np
from datetime import timedelta
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import precision_score, recall_score, roc_auc_score
from dotenv import load_dotenv

warnings.filterwarnings('ignore')
load_dotenv()

# ========== CONFIGURATION ==========
DUNE_API_KEY = os.getenv("DUNE_WHALES_API")
COINGECKO_API_KEY = os.getenv("COINGECKO_API_KEY")

# Create directories
for d in ['data', 'data/price_cache', 'validation', 'backtest', 'models']:
    os.makedirs(d, exist_ok=True)

# API endpoints
QUERIES = {
    "whales": ("6395391", "data/dune_whales_cache.json", "data/whale_ml_ready.csv"),
    "market_intent": ("6385600", "data/dune_intent_cache.json", "data/market_intent_ml_ready.csv")
}

# Date ranges
COINGECKO_START = pd.Timestamp('2017-05-01', tz='UTC')
DUNE_START = pd.Timestamp('2017-10-16', tz='UTC')

# Feature sets
SHORT_FEATURES = [
    'exchange_flow_share', 'net_exchange_flow_ratio', 'whale_exchange_flow_ratio',
    'whale_exchange_asymmetry', 'eth_vol7', 'eth_vol30', 'btc_ret_lag1',
    'btc_ret_lag3', 'eth_btc_corr_30d', 'whale_volume_ratio_delta_3d',
    'exchange_volume_zscore'
]

LONG_FEATURES = [
    'btc_rsi', 'eth_vol7', 'whale_volume_ratio', 'eth_rsi',
    'btc_ret_lag1', 'eth_burned_zscore_90d', 'eth_btc_corr_30d',
    'eth_ret_lag1', 'btc_ret_lag7', 'btc_vol30',
    'whale_volume_ratio_delta_1d', 'whale_volume_ratio_delta_3d',
    'exchange_flow_share', 'net_exchange_flow_ratio',
    'whale_exchange_flow_ratio', 'tx_per_active_zscore_90d'
]

# Trading parameters
SLIPPAGE = 0.0008
FEES = 0.0004

# ========== DATA FETCHING FUNCTIONS ==========
def to_utc(ts):
    """Ensure timestamp is UTC"""
    ts = pd.Timestamp(ts)
    return ts.tz_localize("UTC") if ts.tzinfo is None else ts.tz_convert("UTC")

def fetch_dune(qid, cache_path):
    """Incrementally fetch Dune data"""
    headers = {"x-dune-api-key": DUNE_API_KEY}
    today = pd.Timestamp.now(tz='UTC').normalize()
    yesterday = today - timedelta(1)
    
    if os.path.exists(cache_path):
        with open(cache_path) as f:
            cached = json.load(f)
        df_cached = pd.DataFrame(cached["data"])
        df_cached["block_date"] = pd.to_datetime(df_cached["block_date"], utc=True)
        last_date = pd.to_datetime(cached["last_block_date"], utc=True)
        
        if last_date >= yesterday:
            print(f"✅ {os.path.basename(cache_path)} current ({last_date.date()})")
            return df_cached
    
    # Execute new query
    resp = requests.post(
        f"https://api.dune.com/api/v1/query/{qid}/execute", 
        headers=headers, 
        timeout=30
    ).json()
    eid = resp["execution_id"]
    
    # Wait for completion
    for _ in range(60):
        status = requests.get(
            f"https://api.dune.com/api/v1/execution/{eid}/status", 
            headers=headers
        ).json()["state"]
        if status == "QUERY_STATE_COMPLETED":
            break
        time.sleep(10)
    
    # Get results
    result = requests.get(
        f"https://api.dune.com/api/v1/execution/{eid}/results", 
        headers=headers
    ).json()["result"]["rows"]
    
    df_new = pd.DataFrame(result)
    df_new["block_date"] = pd.to_datetime(df_new["block_date"], utc=True)
    
    # Merge with cached if exists
    if 'df_cached' in locals():
        df = pd.concat([df_cached, df_new]) \
               .drop_duplicates("block_date") \
               .sort_values("block_date") \
               .reset_index(drop=True)
    else:
        df = df_new
    
    # Update cache
    with open(cache_path, "w") as f:
        json.dump({
            "last_block_date": df["block_date"].max().strftime("%Y-%m-%d"),
            "data": json.loads(df.to_json(orient="records"))
        }, f)
    
    return df

def fetch_cg_chunked(cg_id, start, end, key, days=30):
    """Fetch CoinGecko prices in chunks"""
    url = "https://pro-api.coingecko.com/api/v3"
    headers = {"x-cg-pro-api-key": key}
    start_dt, end_dt = to_utc(start), to_utc(end) + pd.Timedelta(days=1)
    all_prices, curr = [], start_dt
    
    while curr < end_dt:
        next_dt = min(curr + pd.Timedelta(days=days), end_dt)
        params = {
            "vs_currency": "usd",
            "from": int(curr.timestamp()),
            "to": int(next_dt.timestamp())
        }
        
        r = requests.get(
            f"{url}/coins/{cg_id}/market_chart/range",
            params=params,
            headers=headers,
            timeout=30
        )
        
        if 'prices' in r.json():
            all_prices.extend(r.json().get("prices", []))
        time.sleep(0.3)
        curr = next_dt
    
    if not all_prices:
        return pd.DataFrame(columns=["date", "price"])
    
    df = pd.DataFrame(all_prices, columns=["timestamp", "price"])
    df["date"] = pd.to_datetime(df["timestamp"], unit="ms", utc=True).dt.floor("D")
    return df.groupby("date")["price"].mean().reset_index()

def get_price(symbol, cg_id, start, end, key):
    """Incrementally fetch price data"""
    cache_path = f"data/price_cache/{symbol}.csv"
    today_utc = pd.Timestamp.utcnow().floor("D")
    yesterday = today_utc - pd.Timedelta(days=1)
    start_dt, end_dt = to_utc(start), min(to_utc(end), yesterday)
    
    if os.path.exists(cache_path):
        df = pd.read_csv(cache_path, parse_dates=["date"])
        df["date"] = df["date"].apply(to_utc)
        
        if df["date"].max() >= end_dt:
            print(f"✅ {symbol.upper()} cache current")
            return df
        
        # Fetch missing data
        fetch_start = df["date"].max() + pd.Timedelta(days=1)
        if fetch_start <= end_dt:
            new_data = fetch_cg_chunked(cg_id, fetch_start, end_dt, key)
            if not new_data.empty:
                df = pd.concat([df, new_data.rename(columns={"price": f"{symbol}_price"})]) \
                       .drop_duplicates("date") \
                       .reset_index(drop=True)
    else:
        df = fetch_cg_chunked(cg_id, start_dt, end_dt, key) \
               .rename(columns={"price": f"{symbol}_price"})
    
    df.to_csv(cache_path, index=False)
    return df

def load_all_data():
    """Load all data sources"""
    print("📊 Loading data sources...")
    
    # Fetch Dune data
    df_whales = fetch_dune(QUERIES["whales"][0], QUERIES["whales"][1])
    df_whales.to_csv(QUERIES["whales"][2], index=False)
    
    time.sleep(0.5)
    
    df_market = fetch_dune(QUERIES["market_intent"][0], QUERIES["market_intent"][1])
    df_market.to_csv(QUERIES["market_intent"][2], index=False)
    
    # Determine date range for prices
    max_date = max(df_whales["block_date"].max(), df_market["block_date"].max())
    
    # Fetch prices
    df_btc = get_price("btc", "bitcoin", COINGECKO_START, max_date, COINGECKO_API_KEY)
    df_eth = get_price("eth", "ethereum", COINGECKO_START, max_date, COINGECKO_API_KEY)
    
    print("✅ All data loaded")
    return df_whales, df_market, df_btc, df_eth

# ========== FEATURE ENGINEERING ==========
def rolling_zscore(s, window=90):
    """Calculate rolling z-score"""
    return ((s - s.rolling(window).mean()) / s.rolling(window).std()).shift(1)

def add_price_features(df, price_col, prefix):
    """Add technical features for a price series"""
    df = df.copy()
    
    # Log returns
    df[f'{prefix}_log_return'] = np.log(df[price_col] / df[price_col].shift(1))
    
    # Lagged returns
    for lag in [1, 3, 7]:
        df[f'{prefix}_ret_lag{lag}'] = df[f'{prefix}_log_return'].shift(lag)
    
    # Volatility
    df[f'{prefix}_vol7'] = df[f'{prefix}_log_return'].rolling(7).std().shift(1)
    df[f'{prefix}_vol30'] = df[f'{prefix}_log_return'].rolling(30).std().shift(1)
    
    # RSI
    returns = df[f'{prefix}_log_return']
    gains = returns.where(returns > 0, 0).rolling(14).mean()
    losses = -returns.where(returns < 0, 0).rolling(14).mean()
    df[f'{prefix}_rsi'] = (100 - (100 / (1 + gains / (losses + 1e-10)))).shift(1)
    
    return df

def engineer_features(df_whales, df_market_intent, df_btc, df_eth):
    """Engineer all features"""
    print("🔧 Engineering features...")
    
    # Merge price data
    df_prices = pd.merge(df_btc, df_eth, on='date', how='outer').sort_values('date')
    
    # Merge with whale data
    df = pd.merge(
        df_whales, 
        df_prices, 
        left_on='block_date', 
        right_on='date', 
        how='left'
    ).drop(columns=['date'])
    
    # Merge with market intent data
    df = pd.merge(
        df, 
        df_market_intent, 
        on='block_date', 
        how='left', 
        suffixes=('', '_intent')
    )
    
    df = df.sort_values('block_date').reset_index(drop=True)
    
    # Add price features
    df = add_price_features(df, 'eth_price', 'eth')
    df = add_price_features(df, 'btc_price', 'btc')
    
    # ETH/BTC ratio features
    df['eth_btc_ratio'] = df['eth_price'] / df['btc_price']
    df['eth_btc_ratio_ma7'] = df['eth_btc_ratio'].rolling(7).mean().shift(1)
    df['eth_btc_corr_30d'] = df['eth_log_return'].shift(1).rolling(30) \
        .corr(df['btc_log_return'].shift(1)).shift(1)
    
    # On-chain z-scores
    zscore_pairs = [
        ('whale_tx_count', 'whale_tx_zscore_90d'),
        ('tx_per_active', 'tx_per_active_zscore_90d'),
        ('eth_burned', 'eth_burned_zscore_90d')
    ]
    
    for raw_col, zscore_col in zscore_pairs:
        if raw_col in df.columns:
            df[zscore_col] = rolling_zscore(df[raw_col], 90)
    
    # Exchange volume z-score
    if 'exchange_volume' in df.columns:
        df['exchange_volume_zscore'] = rolling_zscore(df['exchange_volume'], 90)
    
    # Burn/issuance ratio
    if all(col in df.columns for col in ['eth_burned', 'total_gas_fees']):
        df['burn_issuance_ratio'] = (df['eth_burned'] / (df['total_gas_fees'] + 1e-10)).shift(1)
    
    # Whale volume deltas
    if 'whale_volume_ratio' in df.columns:
        df['whale_volume_ratio_delta_1d'] = df['whale_volume_ratio'].diff(1).shift(1)
        df['whale_volume_ratio_delta_3d'] = df['whale_volume_ratio'].diff(3).shift(1)
    
    # Clean up intermediate columns
    df = df.drop(columns=['eth_log_return', 'btc_log_return'], errors='ignore')
    
    # Save engineered features
    df.to_csv('data/features_engineered.csv', index=False)
    print(f"✅ Features engineered: {len(df.columns)} columns, {len(df)} rows")
    
    return df

# ========== TARGET CREATION ==========
def create_targets_two_tier(df, k=1.5):
    """Create two-tier SHORT labels (crash + breakdown)"""
    print("🎯 Creating two-tier targets...")
    
    df = df.sort_values('block_date').reset_index(drop=True).copy()
    
    # Calculate returns
    df['eth_log_return'] = np.log(df['eth_price'] / df['eth_price'].shift(1))
    df['rolling_vol_30'] = df['eth_log_return'].rolling(30, min_periods=10).std()
    
    # T+2 returns and threshold
    df['return_t2'] = df['eth_log_return'].rolling(2).sum().shift(-2)
    df['threshold_t2'] = df['rolling_vol_30'].rolling(30, min_periods=10).median() * k
    
    # Tier 1: Crash (hard down)
    hard_down = (
        (df['return_t2'] < -df['threshold_t2']) &
        (df['eth_vol7'] > df['eth_vol30']).fillna(False)
    )
    
    # Tier 2: Breakdown (pre-crash)
    exchange_flow_median = df['exchange_flow_share'].rolling(90, min_periods=30).median()
    
    soft_down = (
        (df['eth_ret_lag1'].fillna(0) < 0) &
        (df['btc_ret_lag1'].fillna(0) < 0) &
        (df['whale_volume_ratio_delta_3d'].fillna(0) > 0) &
        (df['exchange_flow_share'] > exchange_flow_median).fillna(False)
    )
    
    # Create targets
    df['target_t2'] = 0
    df.loc[df['return_t2'] > df['threshold_t2'], 'target_t2'] = 1  # UP
    df.loc[hard_down | soft_down, 'target_t2'] = -1  # DOWN (both tiers)
    
    # Create binary targets
    df['y_long_t2'] = (df['target_t2'] == 1).astype(int)
    df['y_short_t2'] = (df['target_t2'] == -1).astype(int)
    
    # Clean up
    df = df.drop(columns=['eth_log_return'], errors='ignore')
    
    # Print distribution
    print("\n📊 Target Distribution (Two-Tier SHORT):")
    for state, label in [(-1, 'DOWN'), (0, 'FLAT'), (1, 'UP')]:
        count = (df['target_t2'] == state).sum()
        percentage = count / len(df) * 100
        print(f"  {label:5s}: {count:4d} ({percentage:5.1f}%)")
    
    hard_count = hard_down.sum()
    soft_count = soft_down.sum()
    total_down = (df['target_t2'] == -1).sum()
    
    print(f"\n  Tier 1 (crash):     {hard_count:4d}")
    print(f"  Tier 2 (breakdown): {soft_count:4d}")
    print(f"  Total DOWN:         {total_down:4d}")
    
    return df

# ========== REGIME DEFINITION ==========
def define_regimes_extended(df):
    """Define trading regimes including R5 distribution regime"""
    print("📈 Defining extended regimes...")
    
    if 'btc_ret_lag1' not in df.columns or 'eth_vol7' not in df.columns:
        df['regime_code'] = 'R0'
        return df
    
    # Standard regimes based on BTC trend and ETH volatility
    btc_trend_7d = df['btc_ret_lag1'].rolling(7).mean()
    df['btc_regime'] = pd.cut(
        btc_trend_7d, 
        bins=[-np.inf, -0.005, 0.005, np.inf], 
        labels=['DOWN', 'FLAT', 'UP']
    )
    
    vol_median = df['eth_vol7'].rolling(180, min_periods=60).median()
    df['vol_regime'] = (df['eth_vol7'] > vol_median).map({True: 'HIGH', False: 'LOW'})
    
    # Combine for standard regimes
    df['regime'] = df['btc_regime'].astype(str) + '_' + df['vol_regime'].astype(str)
    regime_map = {
        'UP_HIGH': 'R1',    # Bull high vol
        'UP_LOW': 'R2',     # Bull low vol
        'DOWN_HIGH': 'R3',  # Bear high vol
        'DOWN_LOW': 'R4',   # Bear low vol
    }
    df['regime_code'] = df['regime'].map(regime_map).fillna('R0')
    
    # R5: Whale distribution regime
    exchange_flow_median = df['exchange_flow_share'].rolling(60, min_periods=20).median()
    
    df['dist_regime'] = (
        (df['whale_volume_ratio_delta_3d'].fillna(0) > 0) &
        (df['exchange_flow_share'] > exchange_flow_median).fillna(False)
    )
    
    # Override with R5 where distribution regime is active
    df.loc[df['dist_regime'], 'regime_code'] = 'R5'
    
    # Print regime distribution
    print("\n📊 Extended Regime Distribution:")
    regime_stats = []
    for code in ['R1', 'R2', 'R3', 'R4', 'R5', 'R0']:
        count = (df['regime_code'] == code).sum()
        if len(df) > 0:
            pct = count / len(df) * 100
            icon = '🟢' if code == 'R1' else ('🔴' if code in ['R3', 'R5'] else '⚪')
            regime_stats.append(f"{icon} {code}: {count:4d} ({pct:5.1f}%)")
    
    # Print in two columns for better readability
    for i in range(0, len(regime_stats), 2):
        row = regime_stats[i:i+2]
        print("  " + " | ".join(row))
    
    return df

# ========== BUILD COMPLETE PIPELINE ==========
def build_pipeline_complete(df_features):
    """
    Create the complete pipeline dataset with features, targets, and regimes
    """
    print("\n" + "="*70)
    print("BUILDING COMPLETE PIPELINE DATASET")
    print("="*70)
    
    # Create targets
    df_with_targets = create_targets_two_tier(df_features)
    
    # Define regimes
    df_complete = define_regimes_extended(df_with_targets)
    
    # Fill NaN values for features
    feature_cols = [col for col in df_complete.columns if col not in 
                   ['block_date', 'target_t2', 'y_long_t2', 'y_short_t2', 
                    'regime_code', 'btc_regime', 'vol_regime', 'regime', 'dist_regime']]
    
    df_complete[feature_cols] = df_complete[feature_cols].fillna(method='ffill').fillna(0)
    
    # Save complete pipeline
    df_complete.to_csv('data/pipeline_complete.csv', index=False)
    
    # Report statistics
    print(f"\n✅ Pipeline complete saved:")
    print(f"   Rows: {len(df_complete)}")
    print(f"   Columns: {len(df_complete.columns)}")
    print(f"   Date range: {df_complete['block_date'].min().date()} to {df_complete['block_date'].max().date()}")
    print(f"   File: data/pipeline_complete.csv")
    
    return df_complete

# ========== PHASE 1: WALK-FORWARD VALIDATION ==========
def walk_forward_r5_short(df):
    """PHASE 1: Walk-forward validation for R5 SHORT"""
    print("\n" + "="*70)
    print("PHASE 1: WALK-FORWARD VALIDATION (R5 SHORT ISOLATION)")
    print("="*70)
    
    splits = [
        {'train_start':'2017-01-01','train_end':'2020-12-31','test_start':'2021-01-01','test_end':'2021-12-31','name':'2021'},
        {'train_start':'2017-01-01','train_end':'2021-12-31','test_start':'2022-01-01','test_end':'2022-12-31','name':'2022'},
        {'train_start':'2017-01-01','train_end':'2022-12-31','test_start':'2023-01-01','test_end':'2023-12-31','name':'2023'},
        {'train_start':'2017-01-01','train_end':'2023-12-31','test_start':'2024-01-01','test_end':'2024-12-31','name':'2024'},
    ]
    
    results = []
    
    for split in splits:
        # Filter data for split
        train_mask = (df['block_date'] >= split['train_start']) & (df['block_date'] <= split['train_end'])
        test_mask = (df['block_date'] >= split['test_start']) & (df['block_date'] <= split['test_end'])
        
        train = df[train_mask].copy()
        test = df[test_mask].copy()
        
        # Filter to R5 regime only
        train_r5 = train[train['regime_code'] == 'R5'].copy()
        test_r5 = test[test['regime_code'] == 'R5'].copy()
        
        if len(train_r5) < 30 or len(test_r5) < 10:
            print(f"⚠️  {split['name']}: Insufficient R5 data")
            continue
        
        # Prepare features
        features = [f for f in SHORT_FEATURES if f in train_r5.columns]
        X_train = train_r5[features].fillna(method='ffill').fillna(0)
        y_train = train_r5['y_short_t2']
        X_test = test_r5[features].fillna(method='ffill').fillna(0)
        y_test = test_r5['y_short_t2']
        
        # Train model
        model = GradientBoostingClassifier(
            n_estimators=150,
            max_depth=4,
            learning_rate=0.05,
            random_state=42
        )
        model.fit(X_train, y_train)
        
        # Predict
        probs = model.predict_proba(X_test)[:, 1]
        preds = (probs > 0.55).astype(int)
        
        # Calculate metrics
        prec = precision_score(y_test, preds, zero_division=0)
        rec = recall_score(y_test, preds, zero_division=0)
        
        # Calculate AUC if we have both classes
        if len(np.unique(y_test)) > 1:
            auc = roc_auc_score(y_test, probs)
        else:
            auc = 0.0
        
        results.append({
            'period': split['name'],
            'precision': prec,
            'recall': rec,
            'auc': auc,
            'train_size': len(train_r5),
            'test_size': len(test_r5),
            'signals': preds.sum(),
            'threshold': 0.55
        })
        
        # Print results for this split
        status = "✅ PASS" if prec >= 0.65 and rec >= 0.50 else "❌ FAIL"
        print(f"\n{split['name']} {status}")
        print(f"  Precision: {prec:.3f} (req: ≥0.65)")
        print(f"  Recall:    {rec:.3f} (req: ≥0.50)")
        print(f"  AUC:       {auc:.3f}")
        print(f"  Signals:   {preds.sum()}/{len(test_r5)}")
        print(f"  R5 days:   {len(test_r5)}")
    
    # Save results
    df_results = pd.DataFrame(results)
    df_results.to_csv('validation/walk_forward_r5.csv', index=False)
    
    # Calculate overall verdict
    passed = ((df_results['precision'] >= 0.65) & (df_results['recall'] >= 0.50)).sum()
    total = len(df_results)
    
    print(f"\n{'='*70}")
    print(f"VERDICT: {passed}/{total} periods passed")
    if passed == total:
        print("✅ R5 SHORT APPROVED FOR DEPLOYMENT")
    elif passed >= total * 0.75:
        print("⚠️  R5 SHORT CONDITIONALLY APPROVED (reduce position size)")
    else:
        print("❌ R5 SHORT FAILED (do not deploy)")
    print(f"{'='*70}")
    
    return df_results

# ========== PHASE 2: PNL SIMULATION ==========
def simulate_pnl(df):
    """PHASE 2: Regime-aware PnL simulation"""
    print("\n" + "="*70)
    print("PHASE 2: REGIME-AWARE PNL SIMULATION")
    print("="*70)
    
    df = df.sort_values('block_date').reset_index(drop=True).copy()
    
    # Calculate future price for exit
    df['price_t2'] = df['eth_price'].shift(-2)
    
    # Filter to R5 regime for training
    df_r5 = df[df['regime_code'] == 'R5'].copy()
    
    if len(df_r5) < 50:
        print("⚠️  Insufficient R5 data for PnL simulation")
        return pd.DataFrame()
    
    # Prepare features
    features = [f for f in SHORT_FEATURES if f in df_r5.columns]
    
    # Train-test split (80/20)
    split_idx = int(len(df_r5) * 0.8)
    X_train = df_r5[features].iloc[:split_idx].fillna(method='ffill').fillna(0)
    y_train = df_r5['y_short_t2'].iloc[:split_idx]
    
    # Train model
    model = GradientBoostingClassifier(
        n_estimators=150,
        max_depth=4,
        learning_rate=0.05,
        random_state=42
    )
    model.fit(X_train, y_train)
    
    # Generate signals for entire dataset
    X_all = df[features].fillna(method='ffill').fillna(0)
    df['signal_prob'] = model.predict_proba(X_all)[:, 1]
    df['signal'] = ((df['signal_prob'] > 0.55) & (df['regime_code'] == 'R5')).astype(int)
    
    # Simulate trades
    trades = []
    cooldown_until = None
    
    for i, row in df.iterrows():
        # Skip if we don't have exit price
        if pd.isna(row['price_t2']) or pd.isna(row['eth_price']):
            continue
        
        # Check cooldown
        if cooldown_until and row['block_date'] <= cooldown_until:
            continue
        
        # Check signal
        if row['signal'] == 1:
            # Calculate returns (SHORT position)
            gross_ret = -(row['price_t2'] - row['eth_price']) / row['eth_price']
            net_ret = gross_ret - (2 * SLIPPAGE) - (2 * FEES)
            
            trades.append({
                'entry_date': row['block_date'],
                'exit_date': row['block_date'] + timedelta(days=2),
                'regime': row['regime_code'],
                'entry_price': row['eth_price'],
                'exit_price': row['price_t2'],
                'gross_ret': gross_ret,
                'net_ret': net_ret,
                'confidence': row['signal_prob'],
                'direction': 'SHORT'
            })
            
            # Apply cooldown
            cooldown_until = row['block_date'] + timedelta(days=1)
    
    # Create trades DataFrame
    if not trades:
        print("⚠️  No trades generated in simulation")
        return pd.DataFrame()
    
    df_trades = pd.DataFrame(trades)
    
    # Calculate performance metrics
    print("\n📊 PnL Attribution by Regime:")
    regime_stats = df_trades.groupby('regime').agg({
        'net_ret': ['count', 'mean', 'sum', 'std', 'max', 'min']
    }).round(4)
    
    print(regime_stats)
    
    # Overall performance
    total_trades = len(df_trades)
    total_return = df_trades['net_ret'].sum()
    win_rate = (df_trades['net_ret'] > 0).mean()
    avg_win = df_trades[df_trades['net_ret'] > 0]['net_ret'].mean()
    avg_loss = df_trades[df_trades['net_ret'] <= 0]['net_ret'].mean()
    
    # Sharpe ratio (annualized)
    if df_trades['net_ret'].std() > 0:
        sharpe = df_trades['net_ret'].mean() / df_trades['net_ret'].std() * np.sqrt(252 / 3)  # 3-day cycles
    else:
        sharpe = 0
    
    print(f"\n📈 Overall Performance:")
    print(f"  Total trades:      {total_trades}")
    print(f"  Total return:      {total_return * 100:.2f}%")
    print(f"  Win rate:          {win_rate * 100:.1f}%")
    print(f"  Average win:       {avg_win * 100:.2f}%")
    print(f"  Average loss:      {avg_loss * 100:.2f}%")
    print(f"  Sharpe (annual):   {sharpe:.2f}")
    print(f"  Max single win:    {df_trades['net_ret'].max() * 100:.2f}%")
    print(f"  Max single loss:   {df_trades['net_ret'].min() * 100:.2f}%")
    
    # Save trades
    df_trades.to_csv('backtest/pnl_simulation.csv', index=False)
    print(f"\n✅ PnL simulation saved: backtest/pnl_simulation.csv")
    
    return df_trades

# ========== PHASE 3: VETO SCORING ==========
def veto_to_score(row):
    """Convert veto conditions to a score"""
    score = 0
    reasons = []
    
    # Positive factors
    if row.get('whale_exchange_flow_ratio', 0) > 0.6:
        score += 2
        reasons.append('whale_exchange_inflow')
    
    if row.get('net_exchange_flow_ratio', 0) < 0:
        score += 1
        reasons.append('net_flow_negative')
    
    if row.get('btc_ret_lag1', 0) < 0:
        score += 1
        reasons.append('btc_down')
    
    if row.get('exchange_volume_zscore', 0) > 0:
        score += 1
        reasons.append('liquidity_high')
    
    if row.get('eth_vol7', 0) > row.get('eth_vol30', 0):
        score += 1
        reasons.append('vol_expanding')
    
    # Negative factors
    if row.get('whale_volume_ratio', 0) > 0.30 and row.get('exchange_volume_zscore', 0) < -0.5:
        score -= 2
        reasons.append('otc_like_flow')
    
    if row.get('btc_ret_lag1', 0) > 0.02:
        score -= 1
        reasons.append('btc_strong_up')
    
    return score, reasons

def build_signal_object(row, model, features, threshold=0.55):
    """Build final signal object with veto scoring"""
    # Prepare features
    X = row[features].fillna(method='ffill').fillna(0).values.reshape(1, -1)
    
    # Get model probability
    prob = model.predict_proba(X)[0, 1] if hasattr(model, 'predict_proba') else 0.5
    
    # Calculate veto score
    score, reasons = veto_to_score(row)
    
    # Determine action based on score
    if score >= 4:
        action = 'SHORT'
    elif score >= 2:
        action = 'WATCH'
    else:
        action = 'NO_TRADE'
    
    # Model probability override
    if prob < threshold:
        action = 'NO_TRADE'
        reasons.append('low_model_confidence')
    
    # Build signal object
    signal = {
        'date': str(row['block_date'].date()),
        'regime': row.get('regime_code', 'R0'),
        'direction': 'SHORT' if action == 'SHORT' else None,
        'score': score,
        'confidence': round(prob, 3),
        'expected_horizon': 'T+2',
        'action': action,
        'reasons': reasons,
        'model_threshold': threshold
    }
    
    return signal

# ========== PHASE 4: POSITION SIZING ==========
def regime_aware_sizing(confidence, regime):
    """Calculate position size based on confidence and regime"""
    # Base size by confidence
    if confidence < 0.55:
        base_size = 0.0
    elif confidence < 0.60:
        base_size = 0.4
    elif confidence < 0.70:
        base_size = 0.7
    else:
        base_size = 1.0
    
    # Regime multiplier
    regime_mult = {
        'R5': 1.5,  # Distribution regime - highest conviction
        'R3': 1.0,  # Bear high vol
        'R1': 0.75, # Bull high vol
        'R0': 0.0   # Unknown/other
    }
    
    multiplier = regime_mult.get(regime, 0.0)
    final_size = base_size * multiplier
    
    # Cap at 1.5x
    return min(final_size, 1.5)

# ========== PHASE 5: STRESS TESTS ==========
def stress_tests(df, df_trades=None):
    """PHASE 5: Mandatory stress tests"""
    print("\n" + "="*70)
    print("PHASE 5: STRESS TESTS")
    print("="*70)
    
    # 1. Bull Market Stress (2020-2021)
    bull_period = df[(df['block_date'] >= '2020-01-01') & (df['block_date'] <= '2021-12-31')]
    bull_r5_days = (bull_period['regime_code'] == 'R5').sum()
    
    if df_trades is not None:
        bull_trades = df_trades[
            (df_trades['entry_date'] >= '2020-01-01') & 
            (df_trades['entry_date'] <= '2021-12-31')
        ]
        bull_trade_count = len(bull_trades)
        bull_pnl = bull_trades['net_ret'].sum() if not bull_trades.empty else 0
    else:
        bull_trade_count = 0
        bull_pnl = 0
    
    print(f"\n1️⃣ Bull Market Stress (2020-2021):")
    print(f"   R5 days:          {bull_r5_days} (expect: low)")
    print(f"   Trades generated: {bull_trade_count} (expect: few)")
    print(f"   Total PnL:        {bull_pnl * 100:.2f}% (expect: low drawdown)")
    
    # 2. Crash Cluster Stress
    crash_periods = [
        ('Mar 2020', '2020-03-01', '2020-03-31'),
        ('Nov 2022', '2022-11-01', '2022-11-30')
    ]
    
    print(f"\n2️⃣ Crash Cluster Stress:")
    for name, start, end in crash_periods:
        crash_mask = (df['block_date'] >= start) & (df['block_date'] <= end)
        crash_r5 = df[crash_mask]
        r5_count = (crash_r5['regime_code'] == 'R5').sum()
        
        if df_trades is not None:
            crash_trades = df_trades[
                (df_trades['entry_date'] >= start) & 
                (df_trades['entry_date'] <= end)
            ]
            crash_pnl = crash_trades['net_ret'].sum() if not crash_trades.empty else 0
            trade_count = len(crash_trades)
        else:
            crash_pnl = 0
            trade_count = 0
        
        print(f"   {name}:")
        print(f"     R5 days:     {r5_count}")
        print(f"     Trades:      {trade_count}")
        print(f"     PnL:         {crash_pnl * 100:.2f}% (expect: positive convexity)")
    
    # 3. Signal Density Check
    print(f"\n3️⃣ Signal Density Check:")
    
    if df_trades is not None and not df_trades.empty:
        df_trades['year'] = df_trades['entry_date'].dt.year
        yearly_trades = df_trades.groupby('year').size()
        
        print("   Yearly trade count:")
        for year, count in yearly_trades.items():
            status = "✅" if count <= 40 else "⚠️ "
            print(f"     {status} {year}: {count} trades")
        
        if yearly_trades.max() > 40:
            print("   ⚠️  WARNING: Overtrading detected")
        else:
            print("   ✅ Acceptable trade density")
    else:
        print("   No trades to analyze density")
    
    print(f"\n{'='*70}")

# ========== PHASE 6: DEPLOYMENT CHECKLIST ==========
def deployment_checklist():
    """PHASE 6: Live deployment checklist"""
    print("\n" + "="*70)
    print("PHASE 6: LIVE DEPLOYMENT CHECKLIST")
    print("="*70)
    
    checks = {
        'Feature lag integrity': os.path.exists('data/features_engineered.csv'),
        'Data freshness': os.path.exists('data/dune_whales_cache.json'),
        'Model versioning': os.path.exists('models'),
        'Daily log structure': os.path.exists('data'),
        'Validation results': os.path.exists('validation/walk_forward_r5.csv'),
        'Backtest results': os.path.exists('backtest/pnl_simulation.csv'),
        'Complete pipeline': os.path.exists('data/pipeline_complete.csv')
    }
    
    print("✅ Automated checks:")
    for check, passed in checks.items():
        print(f"   {'✅' if passed else '❌'} {check}")
    
    print("\n📋 Manual checks required:")
    print("   ⚠️  API circuit breaker implementation")
    print("   ⚠️  Kill-switch logic (3 consecutive losses)")
    print("   ⚠️  Data pipeline monitoring")
    print("   ⚠️  Model retraining schedule")
    
    print(f"\n{'='*70}")

# ========== MAIN PIPELINE EXECUTION ==========
def run_complete_pipeline():
    """Execute the complete 6-phase pipeline"""
    print("\n" + "="*70)
    print("ETH WHALE ALPHA PIPELINE - COMPLETE EXECUTION")
    print("="*70)
    
    # Step 1: Load data
    df_whales, df_market, df_btc, df_eth = load_all_data()
    
    # Step 2: Engineer features
    df_features = engineer_features(df_whales, df_market, df_btc, df_eth)
    
    # Step 3: Build complete pipeline
    df_pipeline = build_pipeline_complete(df_features)
    
    # Step 4: Train final R5 model for signal generation
    print("\n" + "="*70)
    print("TRAINING FINAL R5 SHORT MODEL")
    print("="*70)
    
    df_r5 = df_pipeline[df_pipeline['regime_code'] == 'R5'].copy()
    features = [f for f in SHORT_FEATURES if f in df_r5.columns]
    
    if len(df_r5) < 50:
        print("⚠️  Insufficient R5 data for model training")
        return None, None, None
    
    # Use 80% of data for training
    split_idx = int(len(df_r5) * 0.8)
    X_train = df_r5[features].iloc[:split_idx].fillna(method='ffill').fillna(0)
    y_train = df_r5['y_short_t2'].iloc[:split_idx]
    
    # Train final model
    final_model = GradientBoostingClassifier(
        n_estimators=150,
        max_depth=4,
        learning_rate=0.05,
        random_state=42
    )
    final_model.fit(X_train, y_train)
    
    # Save model
    joblib.dump(final_model, 'models/r5_short_final.pkl')
    print(f"✅ Final model saved: models/r5_short_final.pkl")
    
    # Step 5: Execute all phases
    print("\n" + "="*70)
    print("EXECUTING ALL 6 PHASES")
    print("="*70)
    
    # Phase 1: Walk-forward validation
    wf_results = walk_forward_r5_short(df_pipeline)
    
    # Phase 2: PnL simulation
    trades = simulate_pnl(df_pipeline)
    
    # Phase 3-4: Generate final signal
    latest_data = df_pipeline.iloc[-1].copy()
    signal = build_signal_object(latest_data, final_model, features, threshold=0.55)
    
    # Add position sizing
    signal['position_size'] = regime_aware_sizing(
        signal['confidence'], 
        signal['regime']
    )
    
    print("\n" + "="*70)
    print("FINAL TRADING SIGNAL")
    print("="*70)
    print(json.dumps(signal, indent=2))
    
    # Phase 5: Stress tests
    stress_tests(df_pipeline, trades)
    
    # Phase 6: Deployment checklist
    deployment_checklist()
    
    # Save final signal
    with open('data/latest_signal.json', 'w') as f:
        json.dump(signal, f, indent=2)
    
    print(f"\n✅ Pipeline execution complete!")
    print(f"   Final signal saved: data/latest_signal.json")
    print(f"   Pipeline data: data/pipeline_complete.csv ({len(df_pipeline)} rows)")
    
    return wf_results, trades, signal

# ========== MAIN EXECUTION ==========
if __name__ == "__main__":
    # Run the complete pipeline
    wf_results, trades, final_signal = run_complete_pipeline()
    
    # Summary
    if final_signal:
        print("\n" + "="*70)
        print("SUMMARY")
        print("="*70)
        print(f"Signal: {final_signal['action']}")
        print(f"Regime: {final_signal['regime']}")
        print(f"Confidence: {final_signal['confidence']}")
        print(f"Position Size: {final_signal['position_size']}")
        print(f"Reasons: {', '.join(final_signal['reasons'])}")


ETH WHALE ALPHA PIPELINE - COMPLETE EXECUTION
📊 Loading data sources...


KeyboardInterrupt: 